In [72]:
%pip install meteostat
%pip install scipy
%pip intall matplotlib.pyplot


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
ERROR: unknown command "intall" - maybe you meant "install"
Note: you may need to restart the kernel to use updated packages.


In [73]:
from datetime import date
import meteostat as ms
from meteostat import Point, daily
import pandas as pd
import numpy as np
#from scipy.stats import norm
#import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple


In [74]:
# 1. Read mainland weather
weather = pd.read_csv("../EDA/stations_daily_statistics_mainland.csv", low_memory=False)
weather["date"] = pd.to_datetime(weather["date"])

# 2. Daily station Tavg (same logic as Orly)
# tmean_calc is already (tmin+tmax)/2 but we can be explicit:
weather["tavg"] = (weather["tmin"] + weather["tmax"]) / 2

# 3. Aggregate to DAILY regional average temperature
reg_daily = (
    weather
    .groupby(["region_code", "date"], as_index=False)
    .agg(tavg_reg=("tavg", "mean"))
)

print(reg_daily.head())

   region_code       date  tavg_reg
0           11 2000-01-01    7.5500
1           11 2000-01-02    7.7500
2           11 2000-01-03    6.1625
3           11 2000-01-04    7.9500
4           11 2000-01-05    8.2000


In [75]:
# 4. Daily HDD and CAT per region, using same formulas as Paris/Orly
reg_daily["hdd"] = (18 - reg_daily["tavg_reg"]).clip(lower=0)
reg_daily["cat"] = reg_daily["tavg_reg"].clip(lower=0)

In [76]:
# 5. Add year and month
reg_daily["year"] = reg_daily["date"].dt.year
reg_daily["month"] = reg_daily["date"].dt.month

# 6. Monthly sum of HDD and CAT per region
monthly_reg = (
    reg_daily
    .groupby(["region_code", "year", "month"], as_index=False)
    .agg(
        hdd_points=("hdd", "sum"),
        cat_points=("cat", "sum"),
    )
)

EURO_PER_POINT = 20

monthly_reg["hdd_eur"] = monthly_reg["hdd_points"] * EURO_PER_POINT
monthly_reg["cat_eur"] = monthly_reg["cat_points"] * EURO_PER_POINT

print(monthly_reg.head())

   region_code  year  month  hdd_points  cat_points      hdd_eur      cat_eur
0           11  2000      1  433.475000  133.208333  8669.500000  2664.166667
1           11  2000      2  330.100000  191.900000  6602.000000  3838.000000
2           11  2000      3  311.020833  246.979167  6220.416667  4939.583333
3           11  2000      4  230.429167  309.570833  4608.583333  6191.416667
4           11  2000      5   87.441667  484.754167  1748.833333  9695.083333


In [77]:
region_focus = 24   # change to whatever region code you want

panel_region = (
    monthly_reg[monthly_reg["region_code"] == region_focus]
    .set_index(["year", "month"])
    .sort_index()
)

print(panel_region.head())

            region_code  hdd_points  cat_points      hdd_eur      cat_eur
year month                                                               
2000 1               24  436.370000  129.243333  8727.400000  2584.866667
     2               24  326.420833  195.579167  6528.416667  3911.583333
     3               24  324.050000  233.950000  6481.000000  4679.000000
     4               24  233.764167  306.235833  4675.283333  6124.716667
     5               24   89.375000  474.545000  1787.500000  9490.900000


In [78]:
def hba_backtest(
    monthly_panel: pd.DataFrame,
    product_col: str,
    listed_months: list,
    start_year: int,
    end_year: int,
    min_history_years: int = 10,
) -> pd.DataFrame:
    """
    Out-of-sample HBA backtest for a given product (hdd_eur or cat_eur).

    For each (year, month), predicts the payoff as the mean of past years'
    payoffs for the same month (expanding window), then compares to realized.
    """
    rows = []

    # full available years in data
    all_years = sorted({y for (y, m) in monthly_panel.index})

    for year in range(start_year, end_year + 1):
        if year not in all_years:
            continue

        for month in listed_months:
            if (year, month) not in monthly_panel.index:
                continue

            # Historical sample: all previous years for the same month
            hist_idx = [(y, month) for y in all_years if y < year and (y, month) in monthly_panel.index]
            if len(hist_idx) < min_history_years:
                # Not enough history yet – skip or mark as NaN
                continue

            hist_vals = monthly_panel.loc[hist_idx, product_col].dropna()

            if hist_vals.empty:
                continue

            hba_pred = hist_vals.mean()
            hba_std = hist_vals.std()

            realized = monthly_panel.loc[(year, month), product_col]

            rows.append({
                'year': year,
                'month': month,
                'product': product_col,
                'hba_pred': hba_pred,
                'hba_std_hist': hba_std,
                'realized': realized,
                'error': realized - hba_pred,
                'abs_error': abs(realized - hba_pred),
                'sq_error': (realized - hba_pred) ** 2,
                'n_hist_years': len(hist_vals),
            })

    return pd.DataFrame(rows).set_index(['year', 'month']).sort_index()

# Example: HBA for HDD in winter months for this region
listed_months = [10, 11, 12, 1, 2, 3]   # or whatever you used for Paris
start_year = 2005
end_year = 2024

hba_reg_hdd = hba_backtest(
    monthly_panel=panel_region,
    product_col="hdd_eur",
    listed_months=listed_months,
    start_year=start_year,
    end_year=end_year,
    min_history_years=10,
)

print(hba_reg_hdd.head())

            product     hba_pred  hba_std_hist      realized        error  \
year month                                                                  
2010 1      hdd_eur  8453.893373   1063.060250  10740.559524  2286.666151   
     2      hdd_eur  7206.212857   1037.547615   7939.430556   733.217698   
     3      hdd_eur  6324.404683    735.014861   6735.294841   410.890159   
     10     hdd_eur  3294.795437   1161.034447   4195.327381   900.531944   
     11     hdd_eur  6202.343056    889.597741   6787.079365   584.736310   

              abs_error      sq_error  n_hist_years  
year month                                           
2010 1      2286.666151  5.228842e+06            10  
     2       733.217698  5.376082e+05            10  
     3       410.890159  1.688307e+05            10  
     10      900.531944  8.109578e+05            10  
     11      584.736310  3.419166e+05            10  


# Knn Based

In [79]:
knn_weights = pd.read_csv("../EDA/region_station_KNN_weights_mainland.csv")

weather_knn = weather.merge(knn_weights, on="station_id", how="inner")

# Decide which region_code to use.
# Here we keep the region from the KNN weights file:
weather_knn = weather_knn.rename(columns={"region_code_y": "region_code"})

# (Optional) Drop the other one to avoid confusion
if "region_code_x" in weather_knn.columns:
    weather_knn = weather_knn.drop(columns=["region_code_x"])

# Weighted temperature
weather_knn["weighted_temp"] = weather_knn["tavg"] * weather_knn["weight"]

reg_daily_knn = (
    weather_knn
    .groupby(["region_code", "date"], as_index=False)
    .agg(
        weighted_sum=("weighted_temp", "sum"),
        weight_sum=("weight", "sum"),
    )
)

reg_daily_knn["tavg_reg_knn"] = reg_daily_knn["weighted_sum"] / reg_daily_knn["weight_sum"]

In [80]:
# Add year/month
reg_daily_knn["year"] = reg_daily_knn["date"].dt.year
reg_daily_knn["month"] = reg_daily_knn["date"].dt.month

# Daily HDD and CAT with the SAME formulas as Orly
reg_daily_knn["hdd"] = (18 - reg_daily_knn["tavg_reg_knn"]).clip(lower=0)
reg_daily_knn["cat"] = reg_daily_knn["tavg_reg_knn"].clip(lower=0)

# Monthly sums per region
monthly_reg_knn = (
    reg_daily_knn
    .groupby(["region_code", "year", "month"], as_index=False)
    .agg(
        hdd_points=("hdd", "sum"),
        cat_points=("cat", "sum"),
    )
)

EURO_PER_POINT = 20

monthly_reg_knn["hdd_eur"] = monthly_reg_knn["hdd_points"] * EURO_PER_POINT
monthly_reg_knn["cat_eur"] = monthly_reg_knn["cat_points"] * EURO_PER_POINT

print(monthly_reg_knn.head())

   region_code  year  month  hdd_points  cat_points      hdd_eur      cat_eur
0           11  2000      1  455.891184  110.444214  9117.823680  2208.884285
1           11  2000      2  334.876856  187.123144  6697.537118  3742.462882
2           11  2000      3  347.307218  210.692782  6946.144370  4213.855630
3           11  2000      4  277.258313  262.741687  5545.166253  5254.833747
4           11  2000      5   94.163635  477.414810  1883.272706  9548.296191


In [81]:
region_focus = 24  # change this to any region_code you want to analyze

panel_region = (
    monthly_reg_knn[monthly_reg_knn["region_code"] == region_focus]
    .set_index(["year", "month"])
    .sort_index()
)

print(panel_region.head())

            region_code  hdd_points  cat_points      hdd_eur      cat_eur
year month                                                               
2000 1               24  451.471984   57.881559  9029.439676  1157.631175
     2               24  390.555113  113.444887  7811.102259  2268.897741
     3               24  443.366270  114.633730  8867.325399  2292.674601
     4               24  372.725860  167.274140  7454.517199  3345.482801
     5               24  264.468811  298.786605  5289.376227  5975.732096


In [82]:
# Example: HDD HBA for winter months in this region
listed_months = [10, 11, 12, 1, 2, 3]  
start_year = 2005
end_year = 2024

hba_reg_hdd = hba_backtest(
    monthly_panel=panel_region,
    product_col="hdd_eur",
    listed_months=listed_months,
    start_year=start_year,
    end_year=end_year,
    min_history_years=10,
)

print(hba_reg_hdd.head())

            product     hba_pred  hba_std_hist      realized        error  \
year month                                                                  
2010 1      hdd_eur  9103.474603   1147.073128  10895.838698  1792.364095   
     2      hdd_eur  7831.674545    771.807772   8179.058624   347.384079   
     3      hdd_eur  7648.835684    588.968653   6939.950787  -708.884897   
     10     hdd_eur  4986.418665   1136.185250   4700.176924  -286.241741   
     11     hdd_eur  7228.515456   1220.167469   7382.871792   154.356335   

              abs_error      sq_error  n_hist_years  
year month                                           
2010 1      1792.364095  3.212569e+06            10  
     2       347.384079  1.206757e+05            10  
     3       708.884897  5.025178e+05            10  
     10      286.241741  8.193433e+04            10  
     11      154.356335  2.382588e+04            10  


# Comparing avg and knn

In [83]:
import pandas as pd
import numpy as np

# Merge daily avg & knn temps
daily_comp = reg_daily.merge(
    reg_daily_knn[["region_code", "date", "tavg_reg_knn"]],
    on=["region_code", "date"],
    how="inner"
)

daily_comp["diff"] = daily_comp["tavg_reg"] - daily_comp["tavg_reg_knn"]

# Overall stats
print("=== DAILY TAVG: avg vs knn ===")
print("Overall correlation:",
      daily_comp["tavg_reg"].corr(daily_comp["tavg_reg_knn"]))
print("Mean abs difference:",
      daily_comp["diff"].abs().mean())
print("Max abs difference:",
      daily_comp["diff"].abs().max())

# Per-region summary
daily_region_summary = (
    daily_comp
    .groupby("region_code")
    .agg(
        corr=("tavg_reg", lambda x: x.corr(daily_comp.loc[x.index, "tavg_reg_knn"])),
        mae=("diff", lambda x: x.abs().mean()),
        max_abs_diff=("diff", lambda x: x.abs().max())
    )
)

print("\nPer-region daily comparison:")
print(daily_region_summary.sort_values("corr"))

=== DAILY TAVG: avg vs knn ===
Overall correlation: 0.8928495240556739
Mean abs difference: 1.4965159217907664
Max abs difference: 30.366666666666664

Per-region daily comparison:
                 corr       mae  max_abs_diff
region_code                                  
53           0.811382  1.589559     23.566667
24           0.832191  1.807874     30.366667
32           0.843586  1.916360     25.900000
93           0.845452  2.624047     28.500000
76           0.870892  1.777940     23.552828
94           0.875291  1.371105     28.150000
28           0.877590  1.329183     25.150000
75           0.892579  1.397861     24.373381
44           0.898893  1.670310     26.675000
27           0.941727  1.047634     27.095326
52           0.941870  0.954543     15.955328
11           0.957649  0.851624     14.658887
84           0.967989  1.124460     18.060310


In [84]:

monthly_comp = monthly_reg.merge(
    monthly_reg_knn,
    on=["region_code", "year", "month"],
    suffixes=("_avg", "_knn")
)

# Compare CAT and HDD points (pre-€ is fine)
for col in ["hdd_points", "cat_points"]:
    c_avg = f"{col}_avg"
    c_knn = f"{col}_knn"
    diff_col = f"diff_{col}"
    monthly_comp[diff_col] = monthly_comp[c_avg] - monthly_comp[c_knn]

    print(f"\n=== MONTHLY {col.upper()}: avg vs knn ===")
    print("Overall correlation:",
          monthly_comp[c_avg].corr(monthly_comp[c_knn]))
    print("Mean abs difference:",
          monthly_comp[diff_col].abs().mean())
    print("Max abs difference:",
          monthly_comp[diff_col].abs().max())


=== MONTHLY HDD_POINTS: avg vs knn ===
Overall correlation: 0.958177968241011
Mean abs difference: 33.733431237313766
Max abs difference: 297.9343637043902

=== MONTHLY CAT_POINTS: avg vs knn ===
Overall correlation: 0.9567296292907654
Mean abs difference: 38.147066718196264
Max abs difference: 589.9964413188674


In [85]:
cat_region_summary = (
    monthly_comp
    .groupby("region_code")
    .agg(
        corr_cat=("cat_points_avg",
                  lambda x: x.corr(monthly_comp.loc[x.index, "cat_points_knn"])),
        mae_cat=("diff_cat_points", lambda x: x.abs().mean())
    )
)

print("\nPer-region CAT comparison:")
print(cat_region_summary.sort_values("corr_cat"))


Per-region CAT comparison:
             corr_cat    mae_cat
region_code                     
24           0.858670  63.995349
53           0.914472  47.712760
32           0.950881  52.589873
93           0.957354  77.753881
44           0.960030  36.869401
94           0.964148  37.178767
28           0.967311  34.865106
76           0.968953  32.700623
75           0.979158  30.290130
27           0.986362  17.556268
52           0.987622  24.068052
11           0.987848  19.471441
84           0.988220  20.860216


In [86]:
# For a given region
region_focus = 24

panel_avg = (
    monthly_reg[monthly_reg["region_code"] == region_focus]
    .set_index(["year", "month"])
    .sort_index()
)

panel_knn = (
    monthly_reg_knn[monthly_reg_knn["region_code"] == region_focus]
    .set_index(["year", "month"])
    .sort_index()
)

listed_months = [10, 11, 12, 1, 2, 3]
start_year = 2005
end_year = 2024

hba_avg = hba_backtest(
    panel_avg,
    product_col="hdd_eur",
    listed_months=listed_months,
    start_year=start_year,
    end_year=end_year,
    min_history_years=10,
)

hba_knn = hba_backtest(
    panel_knn,
    product_col="hdd_eur",
    listed_months=listed_months,
    start_year=start_year,
    end_year=end_year,
    min_history_years=10,
)

In [87]:
hba_avg = hba_avg.reset_index()
hba_knn = hba_knn.reset_index()

hba_comp = hba_avg.merge(
    hba_knn,
    on=["year", "month"],
    suffixes=("_avg", "_knn")
)

print("=== HBA comparison (HDD EUR) ===")
print("Mean HBA price avg:",
      hba_comp["hba_pred_avg"].mean())
print("Mean HBA price knn:",
      hba_comp["hba_pred_knn"].mean())

print("RMSE avg:",
      np.sqrt((hba_comp["error_avg"]**2).mean()))
print("RMSE knn:",
      np.sqrt((hba_comp["error_knn"]**2).mean()))

=== HBA comparison (HDD EUR) ===
Mean HBA price avg: 6627.528836594536
Mean HBA price knn: 7338.4141218295645
RMSE avg: 1056.7366144781117
RMSE knn: 1227.5625393216558


# test of which method explains yield better 

In [88]:
# --- CAT based on regional average temperature ---

reg_daily["year"] = reg_daily["date"].dt.year
reg_daily["month"] = reg_daily["date"].dt.month

# Daily CAT contribution
reg_daily["cat_component_avg"] = reg_daily["tavg_reg"].clip(lower=0)

# May–June CAT sum per region-year
cat_reg_avg = (
    reg_daily.loc[reg_daily["month"].isin([5, 6])]
    .groupby(["region_code", "year"], as_index=False)
    .agg(CAT_MJ_avg=("cat_component_avg", "sum"))
)

print(cat_reg_avg.head())

   region_code  year   CAT_MJ_avg
0           11  2000  1016.662500
1           11  2001  1001.558333
2           11  2002   973.808333
3           11  2003  1071.833333
4           11  2004   940.333333


In [89]:
# --- CAT based on KNN-weighted regional temperature ---

reg_daily_knn["year"] = reg_daily_knn["date"].dt.year
reg_daily_knn["month"] = reg_daily_knn["date"].dt.month

reg_daily_knn["cat_component_knn"] = reg_daily_knn["tavg_reg_knn"].clip(lower=0)

cat_reg_knn = (
    reg_daily_knn.loc[reg_daily_knn["month"].isin([5, 6])]
    .groupby(["region_code", "year"], as_index=False)
    .agg(CAT_MJ_knn=("cat_component_knn", "sum"))
)

print(cat_reg_knn.head())

   region_code  year  CAT_MJ_knn
0           11  2000  969.475602
1           11  2001  844.709493
2           11  2002  763.754174
3           11  2003  871.911359
4           11  2004  869.878981


In [90]:
yield_raw = pd.read_csv(
    "../EDA/ble_tendre_hiver_yield_2000_2024_regions_mainland.csv",
    low_memory=False
)

# Drop missing yield
yield_raw = yield_raw.dropna(subset=["yield"]).copy()

# Aggregate to region-year
yield_reg = (
    yield_raw
    .groupby(["reg", "year"], as_index=False)
    .agg(
        prod_region=("prod", "sum"),
        surf_region=("surf", "sum")
    )
)

# Compute region-level yield
yield_reg["yield"] = yield_reg["prod_region"] / yield_reg["surf_region"]

# Rename for consistency
yield_reg = yield_reg.rename(columns={"reg": "region_code"})

print(yield_reg.head())

   region_code  year  prod_region  surf_region      yield
0           11  2011     18283282       241377  75.745750
1           11  2012     19171886       235948  81.254709
2           11  2013     19783804       236064  83.806951
3           11  2014     20556295       238445  86.209797
4           11  2015     21063650       239290  88.025617


In [91]:
# Merge yield with avg CAT
panel_avg = yield_reg.merge(
    cat_reg_avg,
    on=["region_code", "year"],
    how="inner"
)

panel_avg["log_yield"] = np.log(panel_avg["yield"])

# Merge yield with knn CAT
panel_knn = yield_reg.merge(
    cat_reg_knn,
    on=["region_code", "year"],
    how="inner"
)

panel_knn["log_yield"] = np.log(panel_knn["yield"])

In [92]:
import statsmodels.formula.api as smf

# AVG model
model_avg = smf.ols(
    "log_yield ~ CAT_MJ_avg + C(region_code) + C(year)",
    data=panel_avg
).fit(cov_type="cluster", cov_kwds={"groups": panel_avg["region_code"]})

print(model_avg.summary())

                            OLS Regression Results                            
Dep. Variable:              log_yield   R-squared:                       0.863
Model:                            OLS   Adj. R-squared:                  0.840
Method:                 Least Squares   F-statistic:                     767.6
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           2.23e-15
Time:                        12:57:38   Log-Likelihood:                 151.61
No. Observations:                 182   AIC:                            -249.2
Df Residuals:                     155   BIC:                            -162.7
Df Model:                          26                                         
Covariance Type:              cluster                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                3.9477 

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 26, but rank is 12
  warnings.warn('covariance of constraints does not have full '


In [93]:
# KNN model
model_knn = smf.ols(
    "log_yield ~ CAT_MJ_knn + C(region_code) + C(year)",
    data=panel_knn
).fit(cov_type="cluster", cov_kwds={"groups": panel_knn["region_code"]})

print(model_knn.summary())

                            OLS Regression Results                            
Dep. Variable:              log_yield   R-squared:                       0.862
Model:                            OLS   Adj. R-squared:                  0.839
Method:                 Least Squares   F-statistic:                     82.35
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           1.31e-09
Time:                        12:57:38   Log-Likelihood:                 151.15
No. Observations:                 182   AIC:                            -248.3
Df Residuals:                     155   BIC:                            -161.8
Df Model:                          26                                         
Covariance Type:              cluster                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                4.2660 

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 26, but rank is 12
  warnings.warn('covariance of constraints does not have full '


In [94]:
print("AVG beta:", model_avg.params["CAT_MJ_avg"])
print("AVG p-value:", model_avg.pvalues["CAT_MJ_avg"])
print("AVG R2:", model_avg.rsquared)

print("KNN beta:", model_knn.params["CAT_MJ_knn"])
print("KNN p-value:", model_knn.pvalues["CAT_MJ_knn"])
print("KNN R2:", model_knn.rsquared)

AVG beta: 0.00033285448361518744
AVG p-value: 0.6024906073941707
AVG R2: 0.8630483543055116
KNN beta: 1.8397078875813128e-05
KNN p-value: 0.9210021282581715
KNN R2: 0.8623584470717065


In [95]:
# Merge both indices into one panel
panel_both = yield_reg.merge(
    cat_reg_avg,
    on=["region_code", "year"],
    how="inner"
).merge(
    cat_reg_knn,
    on=["region_code", "year"],
    how="inner"
)

panel_both["log_yield"] = np.log(panel_both["yield"])

model_both = smf.ols(
    "log_yield ~ CAT_MJ_avg + CAT_MJ_knn + C(region_code) + C(year)",
    data=panel_both
).fit(cov_type="cluster", cov_kwds={"groups": panel_both["region_code"]})

print(model_both.summary())

                            OLS Regression Results                            
Dep. Variable:              log_yield   R-squared:                       0.863
Model:                            OLS   Adj. R-squared:                  0.839
Method:                 Least Squares   F-statistic:                     1221.
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           1.38e-16
Time:                        12:57:38   Log-Likelihood:                 151.73
No. Observations:                 182   AIC:                            -247.5
Df Residuals:                     154   BIC:                            -157.8
Df Model:                          27                                         
Covariance Type:              cluster                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                3.9622 

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 27, but rank is 12
  warnings.warn('covariance of constraints does not have full '


In [98]:
import numpy as np

# 1. Load raw regional yield file
yield_raw = pd.read_csv(
    "../EDA/ble_tendre_hiver_yield_2000_2024_regions_mainland.csv",
    low_memory=False
)

# Drop rows with missing yield
yield_raw = yield_raw.dropna(subset=["yield"]).copy()

# 2. Aggregate to region-year (prod-weighted yield)
yield_reg = (
    yield_raw
    .groupby(["reg", "year"], as_index=False)
    .agg(
        prod_region=("prod", "sum"),
        surf_region=("surf", "sum")
    )
)

yield_reg["yield"] = yield_reg["prod_region"] / yield_reg["surf_region"]
yield_reg = yield_reg.rename(columns={"reg": "region_code"})

print(yield_reg.head())

   region_code  year  prod_region  surf_region      yield
0           11  2011     18283282       241377  75.745750
1           11  2012     19171886       235948  81.254709
2           11  2013     19783804       236064  83.806951
3           11  2014     20556295       238445  86.209797
4           11  2015     21063650       239290  88.025617
